In [1]:
import torch
import numpy as np

In [6]:
rng = np.random.default_rng(seed=42)
n_samples = 2048
samples = rng.multivariate_normal(
    mean=np.zeros(5),
    cov=np.eye(5),
    size=n_samples
)

samples2 = rng.multivariate_normal(
    mean=np.ones(5) * 10,
    cov=np.eye(5),
    size=n_samples
)

In [7]:
samples - samples2

array([[ -9.23273668, -10.83763035,  -9.13082149,  -9.00809728,
        -13.1189935 ],
       [ -8.95500578,  -9.9026979 , -10.23674622,  -8.64530002,
        -11.52481114],
       [ -8.39610183,  -8.86517542, -11.78275121,  -9.88273244,
        -10.1069776 ],
       ...,
       [ -8.36636578,  -9.18408946, -11.68119547, -11.29749218,
        -10.28169135],
       [ -8.25748911, -11.44546387,  -9.19500964, -11.40774598,
        -10.33588345],
       [ -9.46508715, -11.8911885 ,  -9.89160607,  -9.85025708,
         -9.89838811]], shape=(2048, 5))

In [8]:
def energy_distance(
    X: torch.Tensor, Y: torch.Tensor, eps: float = 1e-8
) -> torch.Tensor:
    # X: [n, d], Y: [m, d]
    n = X.shape[0]
    m = Y.shape[0]
    # pairwise distances
    # use torch.cdist (fast, differentiable)
    d_xy = torch.cdist(X, Y, p=2)  # [n, m]
    d_xx = torch.cdist(X, X, p=2)  # [n, n]
    d_yy = torch.cdist(Y, Y, p=2)  # [m, m]

    exy = d_xy.mean()
    exx = d_xx.mean()
    eyy = d_yy.mean()

    ed2 = 2.0 * exy - exx - eyy
    # numerical safety: ED >= 0
    ed2 = torch.clamp(ed2, min=0.0)
    return torch.sqrt(ed2 + eps)  # return ED (not squared) to match many definitions


In [9]:
energy_distance(torch.tensor(samples, dtype=torch.float32), torch.tensor(samples2, dtype=torch.float32))

tensor(6.2602)